In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)
pd.set_option("display.max_rows", 200)

print("Libraries loaded successfully.")

Libraries loaded successfully.


In [2]:
#Loading DES and UPAG processed data
DES_PATH = "../data/processed/crop_yield/des_apy_2013_14_2022_23.csv"
UPAG_PATH = "../data/processed/crop_yield/upag_2022_23_2024_25.csv"

des = pd.read_csv(DES_PATH)
upag = pd.read_csv(UPAG_PATH)

print("DES shape :", des.shape)
print("UPAg shape:", upag.shape)

print("\nDES columns:")
print(des.columns.tolist())

print("\nUPAg columns:")
print(upag.columns.tolist())

DES shape : (85920, 16)
UPAg shape: (9384, 16)

DES columns:
['state', 'state_code', 'district', 'district_code', 'crop', 'crop_source_name', 'crop_code', 'crop_type', 'season', 'year', 'area_ha', 'production_tonnes', 'yield_tonnes_ha', 'yield_kg_ha', 'yield_quintal_acre', 'data_source']

UPAg columns:
['state', 'state_code', 'district', 'district_code', 'census_code', 'crop', 'crop_source_name', 'year', 'area_lakh_ha', 'area_ha', 'production_lakh_tonnes', 'production_tonnes', 'yield_kg_ha', 'yield_tonnes_ha', 'yield_quintal_acre', 'data_source']


In [3]:
#Selecting 2022-23
TARGET_YEAR = "2022-2023"

des_2022 = des[
    des["year"] == TARGET_YEAR
].copy()

upag_2022 = upag[
    upag["year"] == TARGET_YEAR
].copy()

print("DES 2022-23 :", des_2022.shape)
print("UPAg 2022-23:", upag_2022.shape)

DES 2022-23 : (9742, 16)
UPAg 2022-23: (3128, 16)


In [4]:
#Basic crop info
print("DES crops:")
print(
    des_2022["crop"]
    .value_counts()
    .sort_index()
)

print("\nUPAg crops:")
print(
    upag_2022["crop"]
    .value_counts()
    .sort_index()
)

DES crops:
crop
Arhar/Tur     637
Bajra         572
Gram          548
Groundnut     961
Jowar         652
Maize        1599
Ragi          426
Rice         1578
Soyabean      430
Sugarcane     507
Urad         1269
Wheat         563
Name: count, dtype: int64

UPAg crops:
crop
Maize    782
Rice     782
Urad     782
Wheat    782
Name: count, dtype: int64


In [5]:
#Validation crops
VALIDATION_CROPS = [
    "Rice",
    "Wheat",
    "Maize",
    "Urad"
]

print("Validation crops:")
for crop in VALIDATION_CROPS:
    print("-", crop)

Validation crops:
- Rice
- Wheat
- Maize
- Urad


In [6]:
#Checking DES season values
print("DES seasons for validation crops:\n")

season_distribution = (
    des_2022[
        des_2022["crop"].isin(VALIDATION_CROPS)
    ]
    .groupby(["crop", "season"])
    .agg(
        district_count=("district_code", "nunique"),
        observation_count=("district_code", "size"),
        total_area_ha=("area_ha", "sum"),
        total_production_tonnes=("production_tonnes", "sum")
    )
    .reset_index()
    .sort_values(
        ["crop", "district_count"],
        ascending=[True, False]
    )
)

display(season_distribution)

DES seasons for validation crops:



,crop,season,district_count,observation_count,total_area_ha,total_production_tonnes
1,Maize,Kharif,552,552,7.491378e+06,2.241874e+07
4,Maize,Total,393,393,7.304143e+06,2.852768e+07
2,Maize,Rabi,351,351,2.191371e+06,1.232166e+07
3,Maize,Summer,214,214,5.062171e+05,2.641566e+06
0,Maize,Autumn,74,74,3.144490e+05,8.780856e+05
6,Maize,Winter,10,10,2.540000e+02,6.951000e+02
5,Maize,Whole Year,5,5,8.820000e+01,2.586700e+02
8,Rice,Kharif,485,485,2.855873e+07,8.812963e+07
11,Rice,Total,363,363,3.248316e+07,9.834238e+07
10,Rice,Summer,246,246,3.119536e+06,1.083685e+07


In [ ]:
#Defininf helper functions
def safe_percentage_difference(des_value, upag_value):
    """
    Calculates absolute percentage difference using DES as denominator. Returns NaN when DES value is zero.
    """
    
    des_value = pd.to_numeric(des_value, errors="coerce")
    upag_value = pd.to_numeric(upag_value, errors="coerce")
    
    result = (
        (des_value - upag_value).abs()
        / des_value.abs()
        * 100
    )
    
    result = result.mask(des_value == 0)
    
    return result

In [8]:
def calculate_comparison(
    des_df,
    upag_df,
    des_filter=None,
    crop=None,
    aggregation_name=None
):
    """
    Merge DES and UPAg data and calculate area, production and yield differences.
    """
    
    d = des_df.copy()
    u = upag_df.copy()
    
    if des_filter is not None:
        d = d[des_filter].copy()
    
    if crop is not None:
        d = d[d["crop"] == crop].copy()
        u = u[u["crop"] == crop].copy()
    
    comparison = pd.merge(
        d,
        u,
        on=[
            "state_code",
            "district_code",
            "crop"
        ],
        how="inner",
        suffixes=("_des", "_upag")
    )
    
    if aggregation_name is not None:
        comparison["des_representation"] = aggregation_name
    
    comparison["area_difference"] = (
        comparison["area_ha_des"]
        - comparison["area_ha_upag"]
    )
    
    comparison["production_difference"] = (
        comparison["production_tonnes_des"]
        - comparison["production_tonnes_upag"]
    )
    
    comparison["yield_difference"] = (
        comparison["yield_kg_ha_des"]
        - comparison["yield_kg_ha_upag"]
    )
    
    comparison["area_difference_pct"] = safe_percentage_difference(
        comparison["area_ha_des"],
        comparison["area_ha_upag"]
    )
    
    comparison["production_difference_pct"] = safe_percentage_difference(
        comparison["production_tonnes_des"],
        comparison["production_tonnes_upag"]
    )
    
    comparison["yield_difference_pct"] = safe_percentage_difference(
        comparison["yield_kg_ha_des"],
        comparison["yield_kg_ha_upag"]
    )
    
    return comparison

In [9]:
#Four specific problematic examples
EXAMPLES = [
    ("Maize", "Tamil Nadu", "Thoothukkudi"),
    ("Rice", "Tripura", "West Tripura"),
    ("Urad", "Tamil Nadu", "Mayiladuthurai"),
    ("Wheat", "Jammu And Kashmir", "Budgam")
]

In [ ]:
#DES Examples
for crop, state, district in EXAMPLES:
    
    print(f"DES | {crop} | {state} | {district}")
    
    temp = des_2022[
        (des_2022["crop"] == crop) &
        (des_2022["state"] == state) &
        (des_2022["district"] == district)
    ].copy()
    
    display(
        temp[
            [
                "state",
                "district",
                "crop",
                "season",
                "area_ha",
                "production_tonnes",
                "yield_kg_ha"
            ]
        ]
        .sort_values("season")
        .reset_index(drop=True)
    )


DES | Maize | Tamil Nadu | Thoothukkudi


,state,district,crop,season,area_ha,production_tonnes,yield_kg_ha
0,Tamil Nadu,Thoothukkudi,Maize,Kharif,2.0,19.0,9500.0
1,Tamil Nadu,Thoothukkudi,Maize,Rabi,50003.0,176557.0,3531.0
2,Tamil Nadu,Thoothukkudi,Maize,Total,50005.0,176576.0,3531.0



DES | Rice | Tripura | West Tripura


,state,district,crop,season,area_ha,production_tonnes,yield_kg_ha
0,Tripura,West Tripura,Rice,Autumn,38.0,102.0,2684.0
1,Tripura,West Tripura,Rice,Kharif,176.0,192.0,1091.0
2,Tripura,West Tripura,Rice,Summer,7996.0,28036.0,3506.0
3,Tripura,West Tripura,Rice,Total,23196.0,79725.0,3437.0
4,Tripura,West Tripura,Rice,Winter,14986.0,51395.0,3430.0



DES | Urad | Tamil Nadu | Mayiladuthurai


,state,district,crop,season,area_ha,production_tonnes,yield_kg_ha
0,Tamil Nadu,Mayiladuthurai,Urad,Kharif,12.0,10.0,833.0
1,Tamil Nadu,Mayiladuthurai,Urad,Rabi,24304.0,5309.0,218.0
2,Tamil Nadu,Mayiladuthurai,Urad,Total,24316.0,5319.0,219.0



DES | Wheat | Jammu And Kashmir | Budgam


,state,district,crop,season,area_ha,production_tonnes,yield_kg_ha
0,Jammu And Kashmir,Budgam,Wheat,Rabi,1071.0,2035.1784,1900.0


In [ ]:
#UPAG examples
for crop, state, district in EXAMPLES:
    
    print(f"UPAg | {crop} | {state} | {district}")
    
    temp = upag_2022[
        (upag_2022["crop"] == crop) &
        (upag_2022["state"] == state) &
        (upag_2022["district"] == district)
    ].copy()
    
    display(
        temp[
            [
                "state",
                "district",
                "crop",
                "area_ha",
                "production_tonnes",
                "yield_kg_ha"
            ]
        ]
        .reset_index(drop=True)
    )


UPAg | Maize | Tamil Nadu | Thoothukkudi


,state,district,crop,area_ha,production_tonnes,yield_kg_ha
0,Tamil Nadu,Thoothukkudi,Maize,50000.0,177000.0,3531.0



UPAg | Rice | Tripura | West Tripura


,state,district,crop,area_ha,production_tonnes,yield_kg_ha
0,Tripura,West Tripura,Rice,23000.0,80000.0,3437.0



UPAg | Urad | Tamil Nadu | Mayiladuthurai


,state,district,crop,area_ha,production_tonnes,yield_kg_ha
0,Tamil Nadu,Mayiladuthurai,Urad,24000.0,5000.0,219.0



UPAg | Wheat | Jammu And Kashmir | Budgam


,state,district,crop,area_ha,production_tonnes,yield_kg_ha
0,Jammu And Kashmir,Budgam,Wheat,1000.0,2000.0,1900.0


In [12]:
#Creating DES season wise comparisons
ALL_SEASONS = sorted(
    des_2022["season"]
    .dropna()
    .unique()
)

print("DES seasons:")
print(ALL_SEASONS)

DES seasons:
['Autumn', 'Kharif', 'Rabi', 'Summer', 'Total', 'Whole Year', 'Winter']


In [13]:
season_comparisons = []

for crop in VALIDATION_CROPS:
    
    for season in ALL_SEASONS:
        
        des_subset = des_2022[
            (des_2022["crop"] == crop) &
            (des_2022["season"] == season)
        ].copy()
        
        if des_subset.empty:
            continue
        
        upag_subset = upag_2022[
            upag_2022["crop"] == crop
        ].copy()
        
        comparison = calculate_comparison(
            des_subset,
            upag_subset,
            aggregation_name=season
        )
        
        if not comparison.empty:
            season_comparisons.append(
                comparison
            )

season_comparisons = pd.concat(
    season_comparisons,
    ignore_index=True
)

print(
    "Total season-level matched observations:",
    len(season_comparisons)
)

Total season-level matched observations: 5009


In [14]:
#Summarizing every season
season_comparison_summary = (
    season_comparisons
    .groupby(
        ["crop", "des_representation"]
    )
    .agg(
        matched_districts=(
            "district_code",
            "nunique"
        ),
        
        median_area_diff_pct=(
            "area_difference_pct",
            "median"
        ),
        
        mean_area_diff_pct=(
            "area_difference_pct",
            "mean"
        ),
        
        median_production_diff_pct=(
            "production_difference_pct",
            "median"
        ),
        
        mean_production_diff_pct=(
            "production_difference_pct",
            "mean"
        ),
        
        median_yield_diff_pct=(
            "yield_difference_pct",
            "median"
        ),
        
        mean_yield_diff_pct=(
            "yield_difference_pct",
            "mean"
        )
    )
    .reset_index()
)

display(
    season_comparison_summary
    .sort_values(
        ["crop", "median_area_diff_pct"]
    )
)

,crop,des_representation,matched_districts,median_area_diff_pct,mean_area_diff_pct,median_production_diff_pct,mean_production_diff_pct,median_yield_diff_pct,mean_yield_diff_pct
4,Maize,Total,393,5.060410,24.511288,1.746725,13.328732,0.000000,8.520801e-17
1,Maize,Kharif,552,38.082444,5088.757143,28.354995,2701.436543,0.249004,8.821001e+00
0,Maize,Autumn,74,100.000000,770.928719,292.884906,2026.510471,26.891859,5.398953e+01
5,Maize,Whole Year,5,100.000000,100.000000,100.000000,100.000000,0.000000,0.000000e+00
6,Maize,Winter,10,100.000000,4952.027650,2029.059402,6305.972059,10.838321,1.286528e+01
2,Maize,Rabi,351,203.030303,20538.668156,188.184438,16673.149493,10.691469,1.797155e+01
3,Maize,Summer,214,658.971407,5113.687624,714.627033,5361.115034,17.221027,3.096781e+01
11,Rice,Total,363,0.321960,2.770906,0.106164,1.409153,0.000000,2.202852e-01
8,Rice,Kharif,485,2.540731,111.758602,1.729400,272.771631,0.000000,6.703368e+00
12,Rice,Winter,193,28.886741,51.642490,29.470126,52.526205,2.045944,3.963866e+00


In [15]:
#Percentage of districts within tolerance
def tolerance_summary(
    comparison_df,
    tolerance
):
    
    rows = []
    
    for (crop, representation), group in comparison_df.groupby(
        ["crop", "des_representation"]
    ):
        
        rows.append({
            "crop": crop,
            "representation": representation,
            "matched": len(group),
            
            "area_within_pct": (
                group["area_difference_pct"] <= tolerance
            ).mean() * 100,
            
            "production_within_pct": (
                group["production_difference_pct"] <= tolerance
            ).mean() * 100,
            
            "yield_within_pct": (
                group["yield_difference_pct"] <= tolerance
            ).mean() * 100
        })
    
    return pd.DataFrame(rows)

In [16]:
tolerance_10 = tolerance_summary(
    season_comparisons,
    10
)

display(
    tolerance_10
    .sort_values(
        ["crop", "area_within_pct"],
        ascending=[True, False]
    )
)

,crop,representation,matched,area_within_pct,production_within_pct,yield_within_pct
4,Maize,Total,393,62.595420,77.099237,100.000000
1,Maize,Kharif,552,27.717391,38.043478,76.811594
0,Maize,Autumn,74,10.810811,14.864865,41.891892
2,Maize,Rabi,351,5.698006,7.122507,46.438746
3,Maize,Summer,214,0.467290,1.869159,35.514019
5,Maize,Whole Year,5,0.000000,0.000000,100.000000
6,Maize,Winter,10,0.000000,0.000000,40.000000
11,Rice,Total,363,94.214876,98.071625,99.724518
8,Rice,Kharif,485,63.505155,64.536082,90.515464
12,Rice,Winter,193,30.051813,32.642487,89.119171


In [17]:
#Exact yield agreement
yield_summary = (
    season_comparisons
    .groupby(
        ["crop", "des_representation"]
    )
    .agg(
        matched=("yield_difference", "size"),
        
        exact_yield_matches=(
            "yield_difference",
            lambda x: (x == 0).sum()
        ),
        
        nonzero_yield_differences=(
            "yield_difference",
            lambda x: (x != 0).sum()
        )
    )
    .reset_index()
)

yield_summary["exact_yield_match_pct"] = (
    yield_summary["exact_yield_matches"]
    /
    yield_summary["matched"]
    * 100
)

display(
    yield_summary
    .sort_values(
        ["crop", "exact_yield_match_pct"],
        ascending=[True, False]
    )
)

,crop,des_representation,matched,exact_yield_matches,nonzero_yield_differences,exact_yield_match_pct
5,Maize,Whole Year,5,5,0,100.000000
4,Maize,Total,393,391,2,99.491094
1,Maize,Kharif,552,258,294,46.739130
0,Maize,Autumn,74,6,68,8.108108
2,Maize,Rabi,351,19,332,5.413105
3,Maize,Summer,214,6,208,2.803738
6,Maize,Winter,10,0,10,0.000000
11,Rice,Total,363,361,2,99.449036
8,Rice,Kharif,485,311,174,64.123711
12,Rice,Winter,193,25,168,12.953368


In [18]:
#Investigating DES total
total_des = des_2022[
    (
        des_2022["season"]
        .astype(str)
        .str.strip()
        .str.lower()
        == "total"
    ) &
    des_2022["crop"].isin(VALIDATION_CROPS)
].copy()

print(
    "DES Total observations:",
    total_des.shape
)

display(
    total_des[
        [
            "crop",
            "state",
            "district",
            "area_ha",
            "production_tonnes",
            "yield_kg_ha"
        ]
    ].head(20)
)

DES Total observations: (1099, 16)


,crop,state,district,area_ha,production_tonnes,yield_kg_ha
76200,Maize,Andhra Pradesh,Alluri Sitharama Raju,5644.0,19517.0,3458.0
76204,Rice,Andhra Pradesh,Alluri Sitharama Raju,58705.0,175374.0,2987.0
76208,Urad,Andhra Pradesh,Alluri Sitharama Raju,3023.0,2502.0,828.0
76218,Maize,Andhra Pradesh,Anakapalli,90.0,688.0,7644.0
76224,Rice,Andhra Pradesh,Anakapalli,59406.0,186481.0,3139.0
76228,Urad,Andhra Pradesh,Anakapalli,8857.0,4975.0,562.0
76246,Maize,Andhra Pradesh,Ananthapuramu,27586.0,161542.0,5856.0
76252,Rice,Andhra Pradesh,Ananthapuramu,26995.0,103988.0,3852.0
76259,Urad,Andhra Pradesh,Ananthapuramu,901.0,1105.0,1226.0
76276,Maize,Andhra Pradesh,Annamayya,3262.0,19707.0,6041.0


In [19]:
#Comparing total against UPAG
total_comparisons = []

for crop in VALIDATION_CROPS:
    
    d = total_des[
        total_des["crop"] == crop
    ].copy()
    
    u = upag_2022[
        upag_2022["crop"] == crop
    ].copy()
    
    if d.empty:
        continue
    
    comparison = calculate_comparison(
        d,
        u,
        aggregation_name="Total"
    )
    
    if not comparison.empty:
        total_comparisons.append(
            comparison
        )

total_comparisons = pd.concat(
    total_comparisons,
    ignore_index=True
)

print(
    "Total matched observations:",
    len(total_comparisons)
)

Total matched observations: 1099


In [ ]:
#Summarizing total
total_summary = (
    total_comparisons
    .groupby("crop")
    .agg(
        matched_districts=(
            "district_code",
            "nunique"
        ),
        
        median_area_diff_pct=(
            "area_difference_pct",
            "median"
        ),
        
        mean_area_diff_pct=(
            "area_difference_pct",
            "mean"
        ),
        
        median_production_diff_pct=(
            "production_difference_pct",
            "median"
        ),
        
        mean_production_diff_pct=(
            "production_difference_pct",
            "mean"
        ),
        
        median_yield_diff_pct=(
            "yield_difference_pct",
            "median"
        ),
        
        mean_yield_diff_pct=(
            "yield_difference_pct",
            "mean"
        )
    )
    .reset_index()
)

display(total_summary)

,crop,matched_districts,median_area_diff_pct,mean_area_diff_pct,median_production_diff_pct,mean_production_diff_pct,median_yield_diff_pct,mean_yield_diff_pct
0,Maize,393,5.060410,24.511288,1.746725,13.328732,0.0,8.520801e-17
1,Rice,363,0.321960,2.770906,0.106164,1.409153,0.0,2.202852e-01
2,Urad,338,14.857075,40.060564,26.700305,48.627406,0.0,2.958580e-01
3,Wheat,5,100.000000,84.089101,100.000000,83.059185,0.0,0.000000e+00


In [21]:
#Total tolerance analysis
total_tolerance_10 = (
    total_comparisons
    .groupby("crop")
    .agg(
        matched=("crop", "size"),
        
        area_within_10_pct=(
            "area_difference_pct",
            lambda x: (x <= 10).mean() * 100
        ),
        
        production_within_10_pct=(
            "production_difference_pct",
            lambda x: (x <= 10).mean() * 100
        ),
        
        yield_within_10_pct=(
            "yield_difference_pct",
            lambda x: (x <= 10).mean() * 100
        ),
        
        exact_yield_matches=(
            "yield_difference",
            lambda x: (x == 0).sum()
        )
    )
    .reset_index()
)

total_tolerance_10["exact_yield_match_pct"] = (
    total_tolerance_10["exact_yield_matches"]
    /
    total_tolerance_10["matched"]
    * 100
)

display(total_tolerance_10)

,crop,matched,area_within_10_pct,production_within_10_pct,yield_within_10_pct,exact_yield_matches,exact_yield_match_pct
0,Maize,393,62.595420,77.099237,100.000000,391,99.491094
1,Rice,363,94.214876,98.071625,99.724518,361,99.449036
2,Urad,338,43.491124,36.686391,99.704142,337,99.704142
3,Wheat,5,0.000000,0.000000,100.000000,5,100.000000


In [22]:
#Testing whether annual values are the sum of seasons
SEASONAL_COMPONENTS = [
    "Kharif",
    "Rabi",
    "Summer",
    "Autumn",
    "Winter"
]

seasonal_des = des_2022[
    des_2022["season"].isin(
        SEASONAL_COMPONENTS
    ) &
    des_2022["crop"].isin(
        VALIDATION_CROPS
    )
].copy()

In [23]:
#Aggregating seasonal values by district
seasonal_sum = (
    seasonal_des
    .groupby(
        [
            "state_code",
            "district_code",
            "crop"
        ]
    )
    .agg(
        seasonal_area_sum_ha=(
            "area_ha",
            "sum"
        ),
        
        seasonal_production_sum_tonnes=(
            "production_tonnes",
            "sum"
        )
    )
    .reset_index()
)

display(
    seasonal_sum.head()
)

,state_code,district_code,crop,seasonal_area_sum_ha,seasonal_production_sum_tonnes
0,1,2,Wheat,1071.0,2035.1784
1,1,4,Wheat,15539.0,34100.3444
2,1,5,Urad,165.0,45.7405
3,1,5,Wheat,81659.0,208238.3007
4,1,7,Wheat,48315.0,98802.4642


In [24]:
#Calculating annual yield from summed seasonal values
seasonal_sum["seasonal_yield_kg_ha"] = np.where(
    seasonal_sum["seasonal_area_sum_ha"] > 0,
    (
        seasonal_sum["seasonal_production_sum_tonnes"]
        * 1000
        /
        seasonal_sum["seasonal_area_sum_ha"]
    ),
    np.nan
)

display(
    seasonal_sum.head()
)

,state_code,district_code,crop,seasonal_area_sum_ha,seasonal_production_sum_tonnes,seasonal_yield_kg_ha
0,1,2,Wheat,1071.0,2035.1784,1900.259944
1,1,4,Wheat,15539.0,34100.3444,2194.500573
2,1,5,Urad,165.0,45.7405,277.215152
3,1,5,Wheat,81659.0,208238.3007,2550.096140
4,1,7,Wheat,48315.0,98802.4642,2044.964591


In [25]:
#Comparing seasonal sum with UPAG
seasonal_vs_upag = pd.merge(
    seasonal_sum,
    upag_2022[
        upag_2022["crop"].isin(
            VALIDATION_CROPS
        )
    ],
    on=[
        "state_code",
        "district_code",
        "crop"
    ],
    how="inner"
)

seasonal_vs_upag["area_difference_pct"] = (
    (
        seasonal_vs_upag["seasonal_area_sum_ha"]
        -
        seasonal_vs_upag["area_ha"]
    ).abs()
    /
    seasonal_vs_upag["seasonal_area_sum_ha"].abs()
    * 100
)

seasonal_vs_upag["production_difference_pct"] = (
    (
        seasonal_vs_upag["seasonal_production_sum_tonnes"]
        -
        seasonal_vs_upag["production_tonnes"]
    ).abs()
    /
    seasonal_vs_upag[
        "seasonal_production_sum_tonnes"
    ].abs()
    * 100
)

seasonal_vs_upag["yield_difference_pct"] = (
    (
        seasonal_vs_upag["seasonal_yield_kg_ha"]
        -
        seasonal_vs_upag["yield_kg_ha"]
    ).abs()
    /
    seasonal_vs_upag[
        "seasonal_yield_kg_ha"
    ].abs()
    * 100
)

display(
    seasonal_vs_upag.head()
)

,state_code,district_code,crop,seasonal_area_sum_ha,seasonal_production_sum_tonnes,seasonal_yield_kg_ha,state,district,census_code,crop_source_name,year,area_lakh_ha,area_ha,production_lakh_tonnes,production_tonnes,yield_kg_ha,yield_tonnes_ha,yield_quintal_acre,data_source,area_difference_pct,production_difference_pct,yield_difference_pct
0,1,2,Wheat,1071.0,2035.1784,1900.259944,Jammu And Kashmir,Budgam,2.0,Wheat,2022-2023,0.01,1000.0,0.02,2000.0,1900.0,1.900,7.689039,UPAg,6.629318,1.728517,0.013679
1,1,4,Wheat,15539.0,34100.3444,2194.500573,Jammu And Kashmir,Doda,16.0,Wheat,2022-2023,0.16,16000.0,0.34,34000.0,2195.0,2.195,8.882864,UPAg,2.966729,0.294262,0.022758
2,1,5,Urad,165.0,45.7405,277.215152,Jammu And Kashmir,Jammu,21.0,Urad,2022-2023,0.00,0.0,0.00,0.0,277.0,0.277,1.120981,UPAg,100.000000,100.000000,0.077612
3,1,5,Wheat,81659.0,208238.3007,2550.096140,Jammu And Kashmir,Jammu,21.0,Wheat,2022-2023,0.82,82000.0,2.08,208000.0,2550.0,2.550,10.319500,UPAg,0.417590,0.114437,0.003770
4,1,7,Wheat,48315.0,98802.4642,2044.964591,Jammu And Kashmir,Kathua,7.0,Wheat,2022-2023,0.48,48000.0,0.99,99000.0,2045.0,2.045,8.275834,UPAg,0.651971,0.199930,0.001732


In [26]:
#Summarizing season sum comparison
seasonal_sum_summary = (
    seasonal_vs_upag
    .groupby("crop")
    .agg(
        matched_districts=(
            "district_code",
            "nunique"
        ),
        
        median_area_diff_pct=(
            "area_difference_pct",
            "median"
        ),
        
        mean_area_diff_pct=(
            "area_difference_pct",
            "mean"
        ),
        
        median_production_diff_pct=(
            "production_difference_pct",
            "median"
        ),
        
        mean_production_diff_pct=(
            "production_difference_pct",
            "mean"
        ),
        
        median_yield_diff_pct=(
            "yield_difference_pct",
            "median"
        ),
        
        mean_yield_diff_pct=(
            "yield_difference_pct",
            "mean"
        )
    )
    .reset_index()
)

display(seasonal_sum_summary)

,crop,matched_districts,median_area_diff_pct,mean_area_diff_pct,median_production_diff_pct,mean_production_diff_pct,median_yield_diff_pct,mean_yield_diff_pct
0,Maize,650,7.024592,31.621331,2.796192,20.188441,0.006378,0.008143
1,Rice,675,0.412373,7.861210,0.171285,6.239108,0.007386,0.127209
2,Urad,581,20.918984,46.005276,49.253731,53.944527,0.034568,0.216662
3,Wheat,544,0.780443,26.115373,0.249357,23.274080,0.004593,0.007770


In [27]:
#Comparing all candidate representations
candidate_results = season_comparison_summary.copy()

candidate_results = candidate_results.rename(
    columns={
        "des_representation": "representation"
    }
)

candidate_results["method_type"] = "Individual season"

candidate_total = total_summary.copy()

candidate_total = candidate_total.rename(
    columns={
        "crop": "crop"
    }
)

candidate_total["representation"] = "Total"
candidate_total["method_type"] = "DES Total"

candidate_total = candidate_total[
    [
        "crop",
        "representation",
        "method_type",
        "matched_districts",
        "median_area_diff_pct",
        "mean_area_diff_pct",
        "median_production_diff_pct",
        "mean_production_diff_pct",
        "median_yield_diff_pct",
        "mean_yield_diff_pct"
    ]
]

candidate_season_sum = seasonal_sum_summary.copy()

candidate_season_sum["representation"] = (
    "Sum of Kharif+Rabi+Summer+Autumn+Winter"
)

candidate_season_sum["method_type"] = (
    "Seasonal aggregation"
)

candidate_season_sum = candidate_season_sum[
    [
        "crop",
        "representation",
        "method_type",
        "matched_districts",
        "median_area_diff_pct",
        "mean_area_diff_pct",
        "median_production_diff_pct",
        "mean_production_diff_pct",
        "median_yield_diff_pct",
        "mean_yield_diff_pct"
    ]
]

candidate_results = pd.concat(
    [
        candidate_results[
            [
                "crop",
                "representation",
                "method_type",
                "matched_districts",
                "median_area_diff_pct",
                "mean_area_diff_pct",
                "median_production_diff_pct",
                "mean_production_diff_pct",
                "median_yield_diff_pct",
                "mean_yield_diff_pct"
            ]
        ],
        candidate_total,
        candidate_season_sum
    ],
    ignore_index=True
)

display(
    candidate_results.sort_values(
        [
            "crop",
            "median_area_diff_pct"
        ]
    )
)

,crop,representation,method_type,matched_districts,median_area_diff_pct,mean_area_diff_pct,median_production_diff_pct,mean_production_diff_pct,median_yield_diff_pct,mean_yield_diff_pct
4,Maize,Total,Individual season,393,5.060410,24.511288,1.746725,13.328732,0.000000,8.520801e-17
24,Maize,Total,DES Total,393,5.060410,24.511288,1.746725,13.328732,0.000000,8.520801e-17
28,Maize,Sum of Kharif+Rabi+Summer+Autumn+Winter,Seasonal aggregation,650,7.024592,31.621331,2.796192,20.188441,0.006378,8.142579e-03
1,Maize,Kharif,Individual season,552,38.082444,5088.757143,28.354995,2701.436543,0.249004,8.821001e+00
0,Maize,Autumn,Individual season,74,100.000000,770.928719,292.884906,2026.510471,26.891859,5.398953e+01
5,Maize,Whole Year,Individual season,5,100.000000,100.000000,100.000000,100.000000,0.000000,0.000000e+00
6,Maize,Winter,Individual season,10,100.000000,4952.027650,2029.059402,6305.972059,10.838321,1.286528e+01
2,Maize,Rabi,Individual season,351,203.030303,20538.668156,188.184438,16673.149493,10.691469,1.797155e+01
3,Maize,Summer,Individual season,214,658.971407,5113.687624,714.627033,5361.115034,17.221027,3.096781e+01
11,Rice,Total,Individual season,363,0.321960,2.770906,0.106164,1.409153,0.000000,2.202852e-01


In [28]:
#Automatically identifying best representation
ranking = candidate_results.copy()

ranking["combined_median_score"] = (
    ranking["median_area_diff_pct"]
    +
    ranking["median_production_diff_pct"]
    +
    ranking["median_yield_diff_pct"]
)

ranking = ranking.sort_values(
    [
        "crop",
        "combined_median_score"
    ]
)

display(
    ranking[
        [
            "crop",
            "representation",
            "method_type",
            "matched_districts",
            "median_area_diff_pct",
            "median_production_diff_pct",
            "median_yield_diff_pct",
            "combined_median_score"
        ]
    ]
)

,crop,representation,method_type,matched_districts,median_area_diff_pct,median_production_diff_pct,median_yield_diff_pct,combined_median_score
4,Maize,Total,Individual season,393,5.060410,1.746725,0.000000,6.807135
24,Maize,Total,DES Total,393,5.060410,1.746725,0.000000,6.807135
28,Maize,Sum of Kharif+Rabi+Summer+Autumn+Winter,Seasonal aggregation,650,7.024592,2.796192,0.006378,9.827162
1,Maize,Kharif,Individual season,552,38.082444,28.354995,0.249004,66.686443
5,Maize,Whole Year,Individual season,5,100.000000,100.000000,0.000000,200.000000
2,Maize,Rabi,Individual season,351,203.030303,188.184438,10.691469,401.906210
0,Maize,Autumn,Individual season,74,100.000000,292.884906,26.891859,419.776765
3,Maize,Summer,Individual season,214,658.971407,714.627033,17.221027,1390.819467
6,Maize,Winter,Individual season,10,100.000000,2029.059402,10.838321,2139.897723
11,Rice,Total,Individual season,363,0.321960,0.106164,0.000000,0.428124


In [29]:
#Getting best candidate per crop
best_candidates = (
    ranking
    .groupby("crop")
    .head(1)
    .reset_index(drop=True)
)

display(best_candidates)

,crop,representation,method_type,matched_districts,median_area_diff_pct,mean_area_diff_pct,median_production_diff_pct,mean_production_diff_pct,median_yield_diff_pct,mean_yield_diff_pct,combined_median_score
0,Maize,Total,Individual season,393,5.060410,24.511288,1.746725,13.328732,0.0,8.520801e-17,6.807135
1,Rice,Total,Individual season,363,0.321960,2.770906,0.106164,1.409153,0.0,2.202852e-01,0.428124
2,Urad,Total,Individual season,338,14.857075,40.060564,26.700305,48.627406,0.0,2.958580e-01,41.557380
3,Wheat,Rabi,Individual season,537,0.749780,25.149742,0.239218,22.277536,0.0,2.693632e-04,0.988998


In [ ]:
#Inspecting worst mismatches for each representation
for crop in VALIDATION_CROPS:
    
    print(f"WORST MATCHES — {crop}")
    
    crop_data = season_comparisons[
        season_comparisons["crop"] == crop
    ].copy()
    
    for representation in crop_data[
        "des_representation"
    ].unique():
        
        temp = crop_data[
            crop_data["des_representation"]
            == representation
        ].copy()
        
        if temp.empty:
            continue
        
        print(
            f"\n--- {representation} ---"
        )
        
        display(
            temp[
                [
                    "state_des",
                    "district_des",
                    "area_ha_des",
                    "area_ha_upag",
                    "area_difference_pct",
                    "production_tonnes_des",
                    "production_tonnes_upag",
                    "production_difference_pct",
                    "yield_kg_ha_des",
                    "yield_kg_ha_upag",
                    "yield_difference_pct"
                ]
            ]
            .sort_values(
                "area_difference_pct",
                ascending=False
            )
            .head(5)
        )


WORST MATCHES — Rice

--- Autumn ---


,state_des,district_des,area_ha_des,area_ha_upag,area_difference_pct,production_tonnes_des,production_tonnes_upag,production_difference_pct,yield_kg_ha_des,yield_kg_ha_upag,yield_difference_pct
101,Odisha,Khordha,10.0,82000.0,819900.000000,20.0,244000.0,1.219900e+06,2000.0,2967.0,48.350000
45,Bihar,Patna,12.0,65000.0,541566.666667,23.0,210000.0,9.129435e+05,1917.0,3240.0,69.014085
8,Assam,Charaideo,15.0,35000.0,233233.333333,23.0,88000.0,3.825087e+05,1533.0,2540.0,65.688193
12,Assam,Dhubri,40.0,78000.0,194900.000000,62.0,244000.0,3.934484e+05,1550.0,3103.0,100.193548
173,West Bengal,Purulia,153.0,249000.0,162645.098039,414.0,593000.0,1.431367e+05,2706.0,2381.0,12.010347



--- Kharif ---


,state_des,district_des,area_ha_des,area_ha_upag,area_difference_pct,production_tonnes_des,production_tonnes_upag,production_difference_pct,yield_kg_ha_des,yield_kg_ha_upag,yield_difference_pct
572,Tripura,West Tripura,176.0,23000.0,12968.181818,192.0,80000.0,41566.666667,1091.0,3437.0,215.032081
569,Tripura,Sepahijala,417.0,46000.0,10931.175060,427.0,159000.0,37136.533958,1024.0,3480.0,239.843750
570,Tripura,South Tripura,954.0,39000.0,3988.050314,998.0,132000.0,13126.452906,1046.0,3381.0,223.231358
566,Tripura,Gomati,1771.0,38000.0,2045.680407,1863.0,126000.0,6663.285024,1052.0,3340.0,217.490494
567,Tripura,Khowai,1224.0,23000.0,1779.084967,1404.0,74000.0,5170.655271,1147.0,3197.0,178.727114



--- Rabi ---


...

,state_des,district_des,area_ha_des,area_ha_upag,area_difference_pct,production_tonnes_des,production_tonnes_upag,production_difference_pct,yield_kg_ha_des,yield_kg_ha_upag,yield_difference_pct
4511,Madhya Pradesh,Datia,1.0,32000.0,3.199900e+06,1.000,10000.0,9.999000e+05,1000.0,330.0,67.000000
4543,Maharashtra,Dharashiv,2.0,41000.0,2.049900e+06,1.056,26000.0,2.462021e+06,528.0,629.0,19.128788
4539,Madhya Pradesh,Vidisha,3.0,55000.0,1.833233e+06,3.000,33000.0,1.099900e+06,1000.0,605.0,39.500000
4500,Madhya Pradesh,Alirajpur,3.0,29000.0,9.665667e+05,2.000,15000.0,7.499000e+05,667.0,510.0,23.538231
4537,Madhya Pradesh,Tikamgarh,17.0,159000.0,9.351941e+05,9.000,84000.0,9.332333e+05,529.0,530.0,0.189036



--- Total ---


,state_des,district_des,area_ha_des,area_ha_upag,area_difference_pct,production_tonnes_des,production_tonnes_upag,production_difference_pct,yield_kg_ha_des,yield_kg_ha_upag,yield_difference_pct
4962,Uttar Pradesh,Siddharthnagar,344.0,0.0,100.0,184.0,0.0,100.0,535.0,535.0,0.0
4671,Arunachal Pradesh,Dibang Valley,69.0,0.0,100.0,48.0,0.0,100.0,696.0,696.0,0.0
4672,Arunachal Pradesh,East Kameng,161.0,0.0,100.0,135.0,0.0,100.0,839.0,839.0,0.0
4674,Arunachal Pradesh,Kamle,55.0,0.0,100.0,59.0,0.0,100.0,1073.0,1073.0,0.0
4673,Arunachal Pradesh,East Siang,157.0,0.0,100.0,161.0,0.0,100.0,1025.0,1025.0,0.0



--- Winter ---


,state_des,district_des,area_ha_des,area_ha_upag,area_difference_pct,production_tonnes_des,production_tonnes_upag,production_difference_pct,yield_kg_ha_des,yield_kg_ha_upag,yield_difference_pct
4983,Odisha,Bargarh,4.0,2000.0,49900.000000,2.1,1000.0,47519.047619,525.0,342.0,34.857143
5003,Odisha,Nuapada,147.0,4000.0,2621.088435,23.2,1000.0,4210.344828,158.0,168.0,6.329114
4997,Odisha,Khordha,43.0,1000.0,2225.581395,11.9,0.0,100.000000,277.0,446.0,61.010830
5008,Odisha,Sundargarh,113.0,1000.0,784.955752,34.0,0.0,100.000000,301.0,391.0,29.900332
4996,Odisha,Keonjhar,270.0,2000.0,640.740741,103.4,1000.0,867.117988,383.0,397.0,3.655352


In [31]:
#Specifically investigating the four original examples
example_results = []

for crop, state, district in EXAMPLES:
    
    # DES rows
    d = des_2022[
        (des_2022["crop"] == crop) &
        (des_2022["state"] == state) &
        (des_2022["district"] == district)
    ].copy()
    
    # UPAg row
    u = upag_2022[
        (upag_2022["crop"] == crop) &
        (upag_2022["state"] == state) &
        (upag_2022["district"] == district)
    ].copy()
    
    if u.empty:
        continue
    
    u_row = u.iloc[0]
    
    for _, d_row in d.iterrows():
        
        example_results.append({
            "crop": crop,
            "state": state,
            "district": district,
            "des_season": d_row["season"],
            
            "des_area_ha": d_row["area_ha"],
            "upag_area_ha": u_row["area_ha"],
            
            "des_production_tonnes":
                d_row["production_tonnes"],
            "upag_production_tonnes":
                u_row["production_tonnes"],
            
            "des_yield_kg_ha":
                d_row["yield_kg_ha"],
            "upag_yield_kg_ha":
                u_row["yield_kg_ha"]
        })

example_results = pd.DataFrame(
    example_results
)

example_results["area_diff_pct"] = (
    (
        example_results["des_area_ha"]
        -
        example_results["upag_area_ha"]
    ).abs()
    /
    example_results["des_area_ha"].abs()
    * 100
)

example_results["production_diff_pct"] = (
    (
        example_results["des_production_tonnes"]
        -
        example_results["upag_production_tonnes"]
    ).abs()
    /
    example_results["des_production_tonnes"].abs()
    * 100
)

example_results["yield_diff_pct"] = (
    (
        example_results["des_yield_kg_ha"]
        -
        example_results["upag_yield_kg_ha"]
    ).abs()
    /
    example_results["des_yield_kg_ha"].abs()
    * 100
)

display(
    example_results.sort_values(
        ["crop", "area_diff_pct"]
    )
)

,crop,state,district,des_season,des_area_ha,upag_area_ha,des_production_tonnes,upag_production_tonnes,des_yield_kg_ha,upag_yield_kg_ha,area_diff_pct,production_diff_pct,yield_diff_pct
1,Maize,Tamil Nadu,Thoothukkudi,Rabi,50003.0,50000.0,176557.0000,177000.0,3531.0,3531.0,5.999640e-03,0.250910,0.000000
2,Maize,Tamil Nadu,Thoothukkudi,Total,50005.0,50000.0,176576.0000,177000.0,3531.0,3531.0,9.999000e-03,0.240123,0.000000
0,Maize,Tamil Nadu,Thoothukkudi,Kharif,2.0,50000.0,19.0000,177000.0,9500.0,3531.0,2.499900e+06,931478.947368,62.831579
6,Rice,Tripura,West Tripura,Total,23196.0,23000.0,79725.0000,80000.0,3437.0,3437.0,8.449733e-01,0.344936,0.000000
7,Rice,Tripura,West Tripura,Winter,14986.0,23000.0,51395.0000,80000.0,3430.0,3437.0,5.347658e+01,55.657165,0.204082
5,Rice,Tripura,West Tripura,Summer,7996.0,23000.0,28036.0000,80000.0,3506.0,3437.0,1.876438e+02,185.347410,1.968055
4,Rice,Tripura,West Tripura,Kharif,176.0,23000.0,192.0000,80000.0,1091.0,3437.0,1.296818e+04,41566.666667,215.032081
3,Rice,Tripura,West Tripura,Autumn,38.0,23000.0,102.0000,80000.0,2684.0,3437.0,6.042632e+04,78331.372549,28.055142
9,Urad,Tamil Nadu,Mayiladuthurai,Rabi,24304.0,24000.0,5309.0000,5000.0,218.0,219.0,1.250823e+00,5.820305,0.458716
10,Urad,Tamil Nadu,Mayiladuthurai,Total,24316.0,24000.0,5319.0000,5000.0,219.0,219.0,1.299556e+00,5.997368,0.000000


In [32]:
#Checking whether DES Total is actually an annual aggregate
des_total = des_2022[
    (
        des_2022["season"]
        .astype(str)
        .str.strip()
        .str.lower()
        == "total"
    ) &
    des_2022["crop"].isin(VALIDATION_CROPS)
].copy()

seasonal_components = des_2022[
    des_2022["season"].isin(
        SEASONAL_COMPONENTS
    ) &
    des_2022["crop"].isin(
        VALIDATION_CROPS
    )
].copy()

seasonal_aggregate = (
    seasonal_components
    .groupby(
        [
            "state_code",
            "district_code",
            "crop"
        ]
    )
    .agg(
        seasonal_area=(
            "area_ha",
            "sum"
        ),
        seasonal_production=(
            "production_tonnes",
            "sum"
        )
    )
    .reset_index()
)

In [33]:
#Merging total with seasonal sum
total_vs_seasons = pd.merge(
    des_total[
        [
            "state_code",
            "district_code",
            "crop",
            "area_ha",
            "production_tonnes"
        ]
    ],
    seasonal_aggregate,
    on=[
        "state_code",
        "district_code",
        "crop"
    ],
    how="inner"
)

total_vs_seasons["area_difference_pct"] = (
    (
        total_vs_seasons["area_ha"]
        -
        total_vs_seasons["seasonal_area"]
    ).abs()
    /
    total_vs_seasons["area_ha"].abs()
    * 100
)

total_vs_seasons["production_difference_pct"] = (
    (
        total_vs_seasons["production_tonnes"]
        -
        total_vs_seasons["seasonal_production"]
    ).abs()
    /
    total_vs_seasons[
        "production_tonnes"
    ].abs()
    * 100
)

display(
    total_vs_seasons.head(20)
)

,state_code,district_code,crop,area_ha,production_tonnes,seasonal_area,seasonal_production,area_difference_pct,production_difference_pct
0,28,745,Maize,5644.0,19517.0,5644.0,19517.0,0.0,0.0
1,28,745,Rice,58705.0,175374.0,58705.0,175374.0,0.0,0.0
2,28,745,Urad,3023.0,2502.0,3023.0,2502.0,0.0,0.0
3,28,744,Maize,90.0,688.0,90.0,688.0,0.0,0.0
4,28,744,Rice,59406.0,186481.0,59406.0,186481.0,0.0,0.0
5,28,744,Urad,8857.0,4975.0,8857.0,4975.0,0.0,0.0
6,28,502,Maize,27586.0,161542.0,27586.0,161542.0,0.0,0.0
7,28,502,Rice,26995.0,103988.0,26995.0,103988.0,0.0,0.0
8,28,502,Urad,901.0,1105.0,901.0,1105.0,0.0,0.0
9,28,753,Maize,3262.0,19707.0,3262.0,19707.0,0.0,0.0


In [34]:
#Summarized total v/s seasonal sum
total_vs_seasons_summary = (
    total_vs_seasons
    .groupby("crop")
    .agg(
        matched_districts=(
            "district_code",
            "nunique"
        ),
        
        median_area_difference_pct=(
            "area_difference_pct",
            "median"
        ),
        
        mean_area_difference_pct=(
            "area_difference_pct",
            "mean"
        ),
        
        median_production_difference_pct=(
            "production_difference_pct",
            "median"
        ),
        
        mean_production_difference_pct=(
            "production_difference_pct",
            "mean"
        )
    )
    .reset_index()
)

display(total_vs_seasons_summary)

,crop,matched_districts,median_area_difference_pct,mean_area_difference_pct,median_production_difference_pct,mean_production_difference_pct
0,Maize,393,0.0,1.193580e-16,0.0,2.723992e-16
1,Rice,363,0.0,2.832467e-16,0.0,1.904267e-16
2,Urad,338,0.0,1.013212e-03,0.0,1.013212e-03
3,Wheat,5,0.0,0.000000e+00,0.0,0.000000e+00


In [35]:
#Exporting all investigation results
OUTPUT_DIR = Path(
    "../data/processed/validation"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

season_distribution.to_csv(
    OUTPUT_DIR / "des_season_distribution_2022_23.csv",
    index=False
)

season_comparison_summary.to_csv(
    OUTPUT_DIR / "des_season_comparison_summary.csv",
    index=False
)

tolerance_10.to_csv(
    OUTPUT_DIR / "des_season_tolerance_10pct.csv",
    index=False
)

yield_summary.to_csv(
    OUTPUT_DIR / "des_season_yield_summary.csv",
    index=False
)

total_summary.to_csv(
    OUTPUT_DIR / "des_total_vs_upag_summary.csv",
    index=False
)

seasonal_sum_summary.to_csv(
    OUTPUT_DIR / "des_seasonal_sum_vs_upag_summary.csv",
    index=False
)

candidate_results.to_csv(
    OUTPUT_DIR / "des_upag_candidate_methods.csv",
    index=False
)

best_candidates.to_csv(
    OUTPUT_DIR / "des_upag_best_candidates.csv",
    index=False
)

example_results.to_csv(
    OUTPUT_DIR / "des_upag_example_districts.csv",
    index=False
)

total_vs_seasons_summary.to_csv(
    OUTPUT_DIR / "des_total_vs_season_sum_summary.csv",
    index=False
)

print("All investigation results exported successfully.")

All investigation results exported successfully.


In [ ]:
#Final notebook summary
print("DES ↔ UPAg SEASON INVESTIGATION COMPLETE")

print("\nValidation crops:")
print(", ".join(VALIDATION_CROPS))

print("\nCandidate representations tested:")
print("- Individual DES seasons")
print("- DES Total")
print("- Sum of Kharif + Rabi + Summer + Autumn + Winter")

print("\nImportant:")
print(
    "The automatically ranked candidate is NOT automatically accepted."
)

print(
    "\nReview the candidate_results, best_candidates, "
    "and example_results tables before finalizing the mapping."
)

DES ↔ UPAg SEASON INVESTIGATION COMPLETE

Validation crops:
Rice, Wheat, Maize, Urad

Candidate representations tested:
- Individual DES seasons
- DES Total
- Sum of Kharif + Rabi + Summer + Autumn + Winter

Important:
The automatically ranked candidate is NOT automatically accepted.

Review the candidate_results, best_candidates, and example_results tables before finalizing the mapping.
